# Lab 8: Multi-Agent Workflow (Capstone)

**AI Agent Lab 8 of 8**

---

## What this lab does

This is the capstone lab. You will combine the core ideas from all seven previous labs into a single automated pipeline with three agents working in sequence:

1. **Triage Agent** - reads an incoming email and classifies it (Urgent, Spam, Normal, Support)
2. **Writer Agent** - drafts an appropriate reply based on the classification
3. **Sender Agent** - attempts to send the reply, retries on timeout, and logs the outcome

You will run the pipeline on five different incoming emails and see a clean, step-by-step log showing every hand-off between agents. The lab ends with a discussion of how `fake_llm` would be replaced by a real AI model.

## What you will learn

**Python concepts:**
- How to chain functions together so output of one becomes input of the next
- How to use a list of dictionaries as a shared pipeline log
- How to write self-contained functions (no imports from other notebooks)
- How to format and print a multi-column report table

**Agent concepts:**
- Multi-agent architecture: how specialised agents hand off work
- Pipelines: sequential agent chains vs. parallel processing
- How simulated AI (`fake_llm`) can be swapped for a real LLM
- End-to-end traceability: logging every agent action in a pipeline

## Important things to know before starting

- **All functions are re-defined here.** This notebook is fully self-contained. You do not need to have run any earlier lab notebooks.
- **What is a pipeline?** A pipeline is a sequence of processing steps where each step receives the output of the previous step.
- **What is multi-agent?** Multiple specialised agents each do one job. This is better than one agent doing everything because each agent is easier to test and replace.
- **This lab runs completely offline.** No internet, no API keys.

## How the pipeline works

```
Incoming Email
        |
        v
  AGENT 1 - Triage Agent
  (classifies the email: Urgent / Support / Spam / Normal)
        |
        v
  AGENT 2 - Writer Agent
  (drafts a reply based on the classification)
        |
        v
  AGENT 3 - Sender Agent
  (validates address, sends with retry, logs outcome)
        |
        v
  PIPELINE LOG - full record of all agent actions
```

---
## Step 0: Libraries and Setup

This lab uses **only built-in Python libraries** - nothing to install, no internet needed.

We use:
- `random` - to simulate send outcomes (sent / timeout / invalid)
- `re` - for keyword-based classification and address validation
- `datetime` - to timestamp pipeline log entries
- `textwrap` - to format email bodies for display
- `sys` - to print the Python version

We call `random.seed(99)` so output is repeatable for every student.

In [ ]:
# No installation needed
# Optional: # !pip install nothing

import sys
print('Setup check passed!')
print(f'Python version: {sys.version}')

---
## Importing Libraries

| Library | What it is used for in this lab |
|---|---|
| `random` | Simulates random send outcomes (sent / timeout / invalid) |
| `re` | Keyword detection for classification and address validation |
| `datetime` | Timestamps each pipeline log entry |
| `textwrap` | Wraps long lines in email bodies for display |

In [ ]:
import random    # used to simulate random send outcomes
import re       # used for keyword detection and address validation
import datetime # used to timestamp pipeline log entries
import textwrap # used to format email body text for display

# Set the random seed so every student sees identical output
random.seed(99)

print('Libraries imported and random seed set to 99.')

---
## Step 1: Define the Shared Pipeline Log

**What this block does:** We create `pipeline_execution_log`, a list that stores one dictionary per pipeline run. Each dictionary records which email was processed, what each agent did, and the final outcome. This gives us full traceability.

**Why we need it:** In a real multi-agent system, logging is essential. You need to be able to look back and see exactly what each agent decided and why.

**What to expect as output:** An empty list ready to be filled by the pipeline.

In [ ]:
# The shared log for the entire pipeline.
# Each entry records one complete run through all three agents for one email.
# This list is printed as the final report in Step 7.
pipeline_execution_log = []

# The weighted outcome list for the simulated email sender
# 'sent' is most common, 'timeout' can trigger retries, 'invalid' stops immediately
list_of_simulated_send_outcomes = [
    'sent', 'sent', 'sent', 'sent',
    'failed: server timeout',
    'failed: server timeout',
    'failed: invalid address'
]

# Maximum times the Sender Agent will retry on timeout
maximum_send_retry_attempts = 3

print(f'Pipeline log initialised (empty). Maximum retries: {maximum_send_retry_attempts}')

---
## Step 2: Define Agent 1 - The Triage Agent

**What this block does:** We re-define the triage agent (from Lab 3 ideas) here. `triage_agent()` reads an incoming email and classifies it into one of four categories: Urgent, Support, Spam, or Normal. It uses keyword rules.

**Why we need it:** The triage classification determines what kind of reply the Writer Agent should draft. Without classification, the Writer cannot pick the right tone.

**What to expect as output:** The function is defined and tested.

In [ ]:
# --- AGENT 1: TRIAGE AGENT ---
# Keyword lists for each category (checked in priority order)
urgent_triage_keywords = ['urgent', 'asap', 'immediately', 'critical', 'emergency', 'right now']
support_triage_keywords = ['refund', 'broken', 'not working', 'help', 'issue', 'problem', 'complaint', 'error']
spam_triage_keywords = ['winner', 'free gift', 'click here', 'prize', 'congratulations', 'unsubscribe', 'earn money']


def triage_agent(incoming_email_dict):
    """
    Classifies an incoming email as 'Urgent', 'Support', 'Spam', or 'Normal'.

    Takes in:
      incoming_email_dict - a dictionary with 'sender', 'subject', 'body' keys

    Returns:
      A string: 'Urgent', 'Support', 'Spam', or 'Normal'
    """

    combined_text = incoming_email_dict['subject'] + ' ' + incoming_email_dict['body']
    lowered_combined_text = combined_text.lower()

    # Check each category in priority order
    for keyword in urgent_triage_keywords:
        if keyword in lowered_combined_text:
            return 'Urgent'

    for keyword in support_triage_keywords:
        if keyword in lowered_combined_text:
            return 'Support'

    for keyword in spam_triage_keywords:
        if keyword in lowered_combined_text:
            return 'Spam'

    return 'Normal'


# Quick test
test_email_for_triage = {'sender': 'x@example.com', 'subject': 'URGENT server down', 'body': 'Fix this now!'}
print(f'Triage test: {triage_agent(test_email_for_triage)}')

---
## Step 3: Define Agent 2 - The Writer Agent

**What this block does:** We re-define the writer agent (from Lab 1 ideas) here. `writer_agent()` receives the original email and the triage classification, then uses `fake_llm()` to pick the right reply template and fill it in.

**Why we need it:** The reply must be appropriate for the email type - an Urgent email gets a direct, serious reply; a Spam email gets a brief dismissal; a Normal email gets a friendly acknowledgement.

**What to expect as output:** The fake_llm and writer_agent functions are defined and tested.

In [ ]:
# --- AGENT 2: WRITER AGENT ---
# Reply templates - one per classification category
# SIMULATED AI: In a real system fake_llm() would call a language model.
reply_templates_by_classification = {
    'Urgent': (
        'Dear {sender_first_name},\n\n'
        'We have received your urgent message regarding: "{email_subject}".\n'
        'This has been escalated to our priority response team. '
        'You can expect a detailed response within the next 30 minutes.\n\n'
        'We apologise for any inconvenience caused.\n\n'
        'Regards,\nThe Support Team'
    ),
    'Support': (
        'Dear {sender_first_name},\n\n'
        'Thank you for contacting us about: "{email_subject}".\n'
        'A support ticket has been created for your request. '
        'One of our agents will follow up with you within 24 hours.\n\n'
        'Best regards,\nCustomer Support'
    ),
    'Normal': (
        'Dear {sender_first_name},\n\n'
        'Thank you for your message: "{email_subject}".\n'
        'We have received it and will get back to you shortly.\n\n'
        'Kind regards,\nThe Team'
    ),
    'Spam': (
        'This message has been identified as unsolicited mail.\n'
        'No reply will be sent. The sender has been flagged.'
    )
}


def fake_llm(email_classification, sender_email_address, email_subject):
    """
    SIMULATED AI: Picks a reply template based on classification and fills it in.

    In a real system, this would call a language model API like Gemini or GPT-4.
    Here, we just select and fill a pre-written template.

    Takes in:
      email_classification  - 'Urgent', 'Support', 'Spam', or 'Normal' (string)
      sender_email_address  - the sender's email (to extract first name from)
      email_subject         - the subject of the incoming email

    Returns:
      A complete reply email body as a string.
    """
    # Extract the sender's first name from their email address (e.g. alice@example.com -> Alice)
    name_part = sender_email_address.split('@')[0].split('.')[0]
    sender_first_name = name_part.capitalize()

    template = reply_templates_by_classification.get(
        email_classification,
        reply_templates_by_classification['Normal']
    )

    # Only non-Spam templates have placeholders
    if email_classification != 'Spam':
        return template.format(
            sender_first_name=sender_first_name,
            email_subject=email_subject[:50]
        )
    return template


def writer_agent(incoming_email_dict, triage_classification):
    """
    Drafts a reply email based on the incoming email and triage classification.

    Takes in:
      incoming_email_dict  - dict with 'sender', 'subject', 'body' keys
      triage_classification - 'Urgent', 'Support', 'Spam', or 'Normal' (string)

    Returns:
      A tuple: (reply_subject_string, reply_body_string)
    """
    reply_body = fake_llm(
        email_classification=triage_classification,
        sender_email_address=incoming_email_dict['sender'],
        email_subject=incoming_email_dict['subject']
    )
    reply_subject = f'Re: {incoming_email_dict["subject"]}'
    return (reply_subject, reply_body)


# Quick test
test_reply_subject, test_reply_body = writer_agent(
    {'sender': 'alice@corp.com', 'subject': 'login not working', 'body': 'I cannot log in'},
    'Support'
)
print(f'Writer test -> Subject: {test_reply_subject}')
print(f'Body preview: {test_reply_body[:80]}...')

---
## Step 4: Define Agent 3 - The Sender Agent

**What this block does:** We re-define the sender agent (from Lab 2 ideas) here. `sender_agent()` validates the reply-to address, attempts to send with retries, and returns the final outcome and attempt count.

**Why we need it:** Sending is the final step in the pipeline. If sending fails, the pipeline log records why so you can diagnose the problem.

**What to expect as output:** The validation and sender functions are defined.

In [ ]:
# --- AGENT 3: SENDER AGENT ---

def validate_reply_address(email_address):
    """
    Checks if an email address looks valid (has '@' and a '.' in the domain).

    Takes in:
      email_address - the address to validate (string)

    Returns:
      True if valid, False if not.
    """
    if '@' not in email_address:
        return False
    domain_section = email_address.split('@')[-1]
    return '.' in domain_section


def simulate_one_send_attempt():
    """
    Simulates a single email send attempt.
    SIMULATED SERVICE: In a real system this calls an email API.

    Takes in: nothing
    Returns: one outcome string from list_of_simulated_send_outcomes
    """
    return random.choice(list_of_simulated_send_outcomes)


def sender_agent(reply_to_address, reply_subject, reply_body):
    """
    Validates the address and sends the reply with retry logic.

    Takes in:
      reply_to_address - the email address to send the reply to (string)
      reply_subject    - the subject of the reply email (string)
      reply_body       - the body text of the reply (string)

    Returns:
      A tuple: (final_outcome_string, number_of_attempts_made)
    """

    # Validate the address before attempting to send
    if not validate_reply_address(reply_to_address):
        return ('failed: invalid address (caught by validator)', 0)

    number_of_attempts_made = 0
    final_send_outcome = None

    while number_of_attempts_made < maximum_send_retry_attempts:
        number_of_attempts_made = number_of_attempts_made + 1
        send_result = simulate_one_send_attempt()
        final_send_outcome = send_result

        if send_result == 'sent':
            return (send_result, number_of_attempts_made)
        elif 'invalid' in send_result:
            return (send_result, number_of_attempts_made)
        # Otherwise it was a timeout - loop will retry

    return (final_send_outcome, number_of_attempts_made)


print('Sender agent functions defined.')

---
## Step 5: Build the Multi-Agent Pipeline

**What this block does:** We define `run_multi_agent_pipeline()`. This is the orchestrator - it takes one incoming email, calls the three agents in sequence, prints the hand-off log, and records everything in `pipeline_execution_log`.

**Why we need it:** The orchestrator is what makes this a pipeline. It controls the flow of data between agents and ensures each hand-off is visible.

**What to expect as output:** The function is defined.

In [ ]:
def run_multi_agent_pipeline(incoming_email_dict):
    """
    Runs one incoming email through all three agents in sequence.

    Prints a step-by-step hand-off log and records the result
    in pipeline_execution_log.

    Takes in:
      incoming_email_dict - dict with 'sender', 'subject', 'body' keys

    Returns:
      Nothing - prints to screen and appends to pipeline_execution_log.
    """

    run_timestamp = datetime.datetime.now().strftime('%H:%M:%S')

    print('\n' + '=' * 65)
    print(f'PIPELINE START | {run_timestamp}')
    print(f'Incoming email from : {incoming_email_dict["sender"]}')
    print(f'Subject             : {incoming_email_dict["subject"]}')
    print('=' * 65)

    # --- HAND-OFF 1: Run Triage Agent ---
    print('[AGENT 1 - TRIAGE] Analysing email...')
    email_classification = triage_agent(incoming_email_dict)
    print(f'[AGENT 1 - TRIAGE] Classification: {email_classification}')
    print(f'[AGENT 1 -> AGENT 2] Handing off to Writer Agent with classification={email_classification}')

    # --- HAND-OFF 2: Run Writer Agent ---
    print('[AGENT 2 - WRITER] Drafting reply...')
    reply_subject, reply_body = writer_agent(incoming_email_dict, email_classification)
    print(f'[AGENT 2 - WRITER] Reply subject: {reply_subject}')
    print(f'[AGENT 2 -> AGENT 3] Handing off to Sender Agent with reply ready')

    # --- HAND-OFF 3: Run Sender Agent ---
    # Send the reply back to the original sender
    reply_destination_address = incoming_email_dict['sender']
    print(f'[AGENT 3 - SENDER] Sending reply to: {reply_destination_address}')

    # Skip sending for Spam - do not waste resources
    if email_classification == 'Spam':
        final_send_outcome = 'skipped: email classified as Spam'
        number_of_send_attempts = 0
        print('[AGENT 3 - SENDER] Spam detected - skipping send.')
    else:
        final_send_outcome, number_of_send_attempts = sender_agent(
            reply_to_address=reply_destination_address,
            reply_subject=reply_subject,
            reply_body=reply_body
        )
        print(f'[AGENT 3 - SENDER] Outcome: {final_send_outcome} (attempts: {number_of_send_attempts})')

    print('-' * 65)
    print(f'PIPELINE COMPLETE | Classification: {email_classification} | Send: {final_send_outcome}')

    # Record everything in the pipeline log
    pipeline_execution_log.append({
        'timestamp': run_timestamp,
        'from': incoming_email_dict['sender'],
        'subject': incoming_email_dict['subject'][:40],
        'classification': email_classification,
        'reply_subject': reply_subject[:40],
        'send_outcome': final_send_outcome,
        'send_attempts': number_of_send_attempts
    })


print('run_multi_agent_pipeline() defined.')

---
## Step 6: Define Sample Emails and Run the Pipeline

**What this block does:** We define five realistic incoming emails with different characteristics and run the full pipeline on each one. The random seed is reset before the run.

**Why we need it:** This is the main demo. You will see all three agents working together for each email, with hand-offs printed at every step.

**What to expect as output:** Five complete pipeline runs with step-by-step hand-off logs.

In [ ]:
# Reset the random seed and pipeline log before the full demo
random.seed(99)
pipeline_execution_log.clear()

# Five incoming emails with different characteristics to exercise all pipeline paths
list_of_incoming_emails = [
    {
        'sender': 'alice@techcorp.com',
        'subject': 'Urgent: Production server is down right now',
        'body': 'Our production server crashed at 3 AM. We need this fixed immediately!'
    },
    {
        'sender': 'bob@client.org',
        'subject': 'Refund request for my last order',
        'body': 'Hello, I have a problem with my order and I would like a refund please.'
    },
    {
        'sender': 'promo@spamsite.biz',
        'subject': 'Congratulations! You are our prize winner!',
        'body': 'Click here to claim your free gift. Unsubscribe anytime.'
    },
    {
        'sender': 'carol@university.edu',
        'subject': 'Question about your services',
        'body': 'Hi, I came across your website and would love to learn more about what you offer.'
    },
    {
        'sender': 'not-a-valid-email',
        'subject': 'Test email with bad sender address',
        'body': 'This email has an invalid sender address so the reply cannot be delivered.'
    }
]

print(f'Running pipeline on {len(list_of_incoming_emails)} incoming emails...')

for incoming_email in list_of_incoming_emails:
    run_multi_agent_pipeline(incoming_email)

---
## Step 7: Print the Final Pipeline Report

**What this block does:** We loop over `pipeline_execution_log` and print a formatted summary table showing all five pipeline runs at a glance. This is the final report the system would deliver to a human supervisor.

**Why we need it:** After all agents have run, the supervisor needs a clear summary - not thousands of lines of logs. The report condenses everything.

**What to expect as output:** A formatted table with one row per email processed.

In [ ]:
# Count outcomes for the summary
total_successfully_sent = sum(1 for entry in pipeline_execution_log if entry['send_outcome'] == 'sent')
total_skipped_as_spam = sum(1 for entry in pipeline_execution_log if 'Spam' in entry['classification'])
total_failed = sum(1 for entry in pipeline_execution_log
                   if 'failed' in entry['send_outcome'])

print('=' * 70)
print('MULTI-AGENT PIPELINE - FINAL REPORT')
print('=' * 70)
print(f'{"From":<25} {"Class":<10} {"Send Outcome":<30} {"Tries"}')
print('-' * 70)

for log_entry in pipeline_execution_log:
    sender_short = log_entry['from'][:23]
    outcome_short = log_entry['send_outcome'][:28]
    print(f"{sender_short:<25} {log_entry['classification']:<10} {outcome_short:<30} {log_entry['send_attempts']}")

print('=' * 70)
print('SUMMARY')
print(f'  Emails processed  : {len(pipeline_execution_log)}')
print(f'  Successfully sent : {total_successfully_sent}')
print(f'  Skipped (Spam)    : {total_skipped_as_spam}')
print(f'  Failed to send    : {total_failed}')
print('=' * 70)

---
## Step 8: Discussion - Replacing fake_llm with a Real Model

This is a discussion cell (no code to run). Read it carefully - it explains how everything you have learned connects to real AI systems.

### How fake_llm connects to a real language model

Throughout all eight labs, we used `fake_llm()` as a stand-in for a real AI language model. When internet access is available, replacing `fake_llm` is straightforward. Here is the idea:

**Current fake_llm (what we have):**
```python
def fake_llm(email_classification, sender_email_address, email_subject):
    # Pick from a hardcoded template dictionary
    template = reply_templates_by_classification[email_classification]
    return template.format(...)
```

**Real LLM version (what you would write with internet access):**
```python
# This is pseudocode - not for running here
# import google.generativeai as genai
# genai.configure(api_key='YOUR_API_KEY')
# model = genai.GenerativeModel('gemini-pro')

def real_llm(email_classification, sender_email_address, email_subject):
    prompt = (
        f'Write a professional email reply. '
        f'The email is classified as: {email_classification}. '
        f'The original subject was: {email_subject}. '
        f'Address the sender by their first name from: {sender_email_address}. '
        f'Keep it under 100 words.'
    )
    response = model.generate_content(prompt)
    return response.text
```

**Key insight:** The rest of the pipeline does not change at all. The triage agent, sender agent, and pipeline orchestrator stay exactly the same. Only `fake_llm` is replaced. This is the power of designing agents as separate, replaceable components.

### What makes a good AI agent system?

After eight labs, here are the core principles you have learned:

| Principle | Where you saw it |
|---|---|
| Separation of concerns | Each agent does one job (Labs 1-8) |
| Simulated AI first | Start with fake_llm, replace later (all labs) |
| Retry and resilience | Sender agent with retry loop (Labs 2, 8) |
| Memory and state | Chatbot memory, task list (Labs 4, 6) |
| Tool use | Calculator, weather, clock (Lab 5) |
| Pipeline composition | Multi-agent hand-offs (Lab 8) |
| Audit logging | Delivery log, pipeline log (Labs 2, 8) |
| Graceful fallback | Unknown requests, invalid addresses (all labs) |

---
## Summary

You completed the **Multi-Agent Workflow Capstone**, combining triage, writing, and sending into a single automated pipeline with full logging.

### Key takeaways
- **Pipelines chain agents:** Each agent does one job and passes its output to the next. This makes the system modular and easier to debug.
- **The orchestrator controls flow:** `run_multi_agent_pipeline()` decides the order of agent calls and handles special cases (like skipping Spam).
- **Logging enables observability:** The pipeline log gives you a complete audit trail without reading thousands of lines of output.
- **fake_llm is a swap-in:** The entire system is ready for a real AI model. Only one function needs to change.
- **You have now built the foundation of real AI agent systems.** Professional agent frameworks like LangChain and Google's Agent Development Kit use these same ideas at a larger scale.

### Congratulations!

You have completed all eight AI Agents labs. You now understand how to design, build, and compose AI agents using only Python's standard library - and how to scale up to real AI models when you have access to the internet.

---

## Try It Yourself

1. **Add a fourth agent:** Design a 'Translator Agent' that detects whether the incoming email is in a language other than English (simple heuristic: check for common non-English words) and adds a note to the reply saying 'Original message detected in [language]'. Insert it between the Triage Agent and the Writer Agent.

2. **Retry report:** In the final pipeline report, add a column showing how many extra retries were needed. Calculate the total retries across all emails and print it in the summary footer.

3. **Priority queue:** Instead of processing emails in the order they arrive, sort `list_of_incoming_emails` so Urgent emails are processed first. Use the triage classification to sort before running the pipeline. Hint: run a quick triage pass first, sort by classification, then run the full pipeline.